# **Checking a CATE estimate: BLP, GATES, CLAN and RATE**

Practice for the module [“Heterogeneous Effects”](https://ai-interpretability.school), lesson “How to read the estimate and where to go wrong”, steps “Is there any heterogeneity: BLP, GATES and CLAN” and “RATE and the TOC curve”.

Calibration by quartiles and the Qini coefficient from the notebook “Heterogeneous effects” show whether a model ranks objects by effect, but without intervals and without a hypothesis test. Here we build the formal checks by hand:

1. BLP ([Chernozhukov, Demirer, Duflo and Fernández-Val, 2017](https://arxiv.org/abs/1712.04802)): is there heterogeneity, and does the model's prediction carry it;
2. GATES: the average effect in five groups by the model's prediction, with intervals;
3. CLAN: how the objects of the groups with the smallest and the largest effect differ;
4. RATE and the TOC curve ([Yadlowsky et al., 2021](https://arxiv.org/abs/2111.07966)): how well the model's prediction picks whom to serve first;
5. where it breaks: an effect without heterogeneity and a check on the same data the model was trained on;
6. the library call `econml.validate.DRTester` and how it differs from the papers.

All checks share one design: the effect model is trained on one part of the sample (auxiliary) and checked on another (main), where its prediction is just a feature fixed in advance.

The notebook runs on a CPU; the whole run takes about a minute.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from econml.dml import CausalForestDML
from econml.validate import DRTester

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # spurious numpy warnings on some processors
SEED = 0
P = 0.5                                                                 # treatment probability in the experiment
pd.set_option('display.width', 140)

## 1. Data, the model's prediction and doubly robust scores

A randomized experiment: 4000 objects, five features $X\sim\mathcal N(0, I)$, treatment for a random half ($p=0.5$). The effect is $\tau(x)=1+1.5\,x_0$, the outcome without treatment is $2x_1+x_2^2$, and the noise has standard deviation 2 — noisier than in the notebook “Heterogeneous effects”, so that the intervals are visible.

The first 2000 objects are the auxiliary part $A$: on it we fit the causal forest, whose prediction $S(x)$ is the “proxy” of the effect in the paper's terms, and an outcome model without treatment $B(x)$. The second 2000 are the main part $M$: all checks happen there. RATE also needs doubly robust effect scores $\hat\Gamma_i$ on the main part, with outcome models from the other folds of the main part; the propensity is known, $p=0.5$.

In [ ]:
def simulate(n, r, heterogeneous=True):
    Xs = r.normal(size=(n, 5))
    ts = 1 + 1.5 * Xs[:, 0] if heterogeneous else np.ones(n)
    Ts = r.binomial(1, P, n)
    Ys = 2 * Xs[:, 1] + Xs[:, 2] ** 2 + Ts * ts + 2 * r.normal(size=n)
    return Xs, Ts, Ys, ts


def gb():
    return GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=SEED)


def proxies(Xs, Ts, Ys, fit, ev):
    # forest and outcome model are fitted on fit; S and B are predicted on ev
    cf = CausalForestDML(model_y=gb(), model_t=LogisticRegression(), discrete_treatment=True,
                         n_estimators=500, min_samples_leaf=20, random_state=SEED).fit(Ys[fit], Ts[fit], X=Xs[fit])
    base = gb().fit(Xs[fit][Ts[fit] == 0], Ys[fit][Ts[fit] == 0])
    return cf, cf.effect(Xs[ev]), base.predict(Xs[ev])


def aipw_scores(Xa, Ta, Ya):
    G = np.zeros(len(Ya))
    for a, b in KFold(5, shuffle=True, random_state=SEED).split(Xa):
        m1 = gb().fit(Xa[a][Ta[a] == 1], Ya[a][Ta[a] == 1]).predict(Xa[b])
        m0 = gb().fit(Xa[a][Ta[a] == 0], Ya[a][Ta[a] == 0]).predict(Xa[b])
        G[b] = m1 - m0 + Ta[b] * (Ya[b] - m1) / P - (1 - Ta[b]) * (Ya[b] - m0) / (1 - P)
    return G


X, T, Y, tau = simulate(4000, np.random.default_rng(SEED))
A, M = np.arange(2000), np.arange(2000, 4000)
cf, S, B = proxies(X, T, Y, A, M)
G = aipw_scores(X[M], T[M], Y[M])
print('true average effect on M:', round(tau[M].mean(), 3), '| mean of Gamma:', round(G.mean(), 3))
print('correlation of the prediction S with the true effect on M:', round(np.corrcoef(S, tau[M])[0, 1], 3))

## 2. BLP: is there heterogeneity

The best linear predictor of the effect given the model's prediction: $\mathrm{BLP}[\tau(X)\mid S]=\beta_1+\beta_2\,(S-\mathbb E S)$, where $\beta_1=\mathbb E\,\tau(X)$ is the average effect and $\beta_2=\mathrm{Cov}(\tau(X), S)/\mathrm{Var}(S)$. If the prediction is perfect, $\beta_2=1$; if it is pure noise or there is no heterogeneity, $\beta_2=0$. Rejecting $\beta_2=0$ means two things at once: there is heterogeneity, and the prediction carries it.

$\tau(X)$ is not observed, but the coefficients are estimated by a weighted regression of the outcome on the main part (strategy A of the paper):

$$Y_i=\alpha_0+\alpha_1B(X_i)+\beta_1\,(T_i-p)+\beta_2\,(T_i-p)\,(S_i-\bar S)+\varepsilon_i$$

with weights $1/(p(1-p))$; with a constant propensity the weights are equal, and this is ordinary least squares. The term $\alpha_1B(X_i)$ reduces noise but is not needed for unbiasedness: the factor $(T_i-p)$ makes the regressors of $\beta_1$ and $\beta_2$ orthogonal to any function of the features.

**Task 1. Write a function `blp(Ym, Tm, S, B)` that runs this regression (`sm.WLS(..., weights=...).fit(cov_type='HC1')`) and returns the estimates of $\beta_1$, $\beta_2$ and their standard errors. Compute them on the main part. What is $\beta_2$, and is the hypothesis $\beta_2=0$ rejected?**

In [ ]:
# Your code here

**Your answer:**

## 3. GATES: the average effect in groups by the prediction

Split the main part into $K=5$ groups by quantiles of the prediction $S$: $G_1$ is the fifth with the smallest prediction, $G_5$ the fifth with the largest. GATES is the average effect in each group, $\gamma_k=\mathbb E[\tau(X)\mid G_k]$. The same regression estimates it, only instead of one term with $(T_i-p)$ there is one per group:

$$Y_i=\alpha_0+\alpha_1B(X_i)+\sum_{k=1}^K\gamma_k\,(T_i-p)\,\mathbb 1[i\in G_k]+\varepsilon_i.$$

This is the calibration by quantiles from the lesson, but with intervals and with an adjustment for outcome noise. If the model ranks the effect meaningfully, $\gamma_k$ grow with $k$, and the difference $\gamma_K-\gamma_1$ is significantly above zero.

**Task 2. Write a function `gates(Ym, Tm, S, B, K=5)` that returns the estimates $\gamma_1,\dots,\gamma_K$, their standard errors and the group of every object (groups are `pd.qcut(S, K, labels=False)`). Print a table: $\gamma_k$, the standard error and the mean true effect in the group. What is the difference $\gamma_5-\gamma_1$?**

In [ ]:
# Your code here

## 4. CLAN: how the extreme groups differ

When GATES shows heterogeneity, the natural question is who these objects are. CLAN compares the feature means in the group with the smallest effect $G_1$ and with the largest $G_K$: $\delta_1=\mathbb E[g(X)\mid G_1]$, $\delta_K=\mathbb E[g(X)\mid G_K]$. These are just means of observed features, and the interval for the difference is the usual two-sample one.

**Task 3. For each of the five features print the mean in $G_1$, the mean in $G_5$, their difference and its standard error $\sqrt{s_1^2/n_1+s_5^2/n_5}$. On which feature do the extreme groups differ the most, and by how much?**

In [ ]:
# Your code here

**Your answer:**

## 5. RATE and the TOC curve

BLP and GATES ask whether there is heterogeneity. RATE asks something else: if objects are served in decreasing order of the prediction $S$, how much larger is the effect among the first ones than on average. The TOC curve (targeting operator characteristic):

$$\mathrm{TOC}(u)=\mathbb E\bigl[\tau(X)\mid S\text{ in the top share }u\bigr]-\mathbb E\,\tau(X),\qquad u\in(0,1].$$

$\mathrm{TOC}(1)=0$; for a prediction unrelated to the effect the curve is near zero everywhere. RATE is a weighted area under the curve: $\theta_\alpha=\int_0^1\alpha(u)\,\mathrm{TOC}(u)\,du$. With $\alpha(u)=1$ it is the AUTOC, with $\alpha(u)=u$ the Qini coefficient in the normalization of Yadlowsky et al. The estimate substitutes doubly robust scores $\hat\Gamma_i$ for $\tau(X_i)$: $\widehat{\mathrm{TOC}}(j/n)=\frac1j\sum_{k\le j}\hat\Gamma_{(k)}-\bar\Gamma$, where $\hat\Gamma_{(k)}$ is the score of the object ranked $k$-th by decreasing $S$. The authors suggest computing the standard error with a half-sample bootstrap: the estimate is recomputed on random halves of the sample, and the standard deviation across repeats is divided by $\sqrt2$.

**Task 4. Write a function `rate(S, G)` that returns the curve $\widehat{\mathrm{TOC}}$ at the points $u=j/n$, the AUTOC and the Qini (the means of $\widehat{\mathrm{TOC}}$ and of $u\cdot\widehat{\mathrm{TOC}}$ over the points). Compute them on the main part, and their standard errors with a bootstrap over 200 random halves (`np.random.default_rng(SEED)`, `choice(n, n // 2, replace=False)`). Draw the TOC curve. What is the AUTOC, and does it differ from zero?**

In [ ]:
# Your code here

**Your answer:**

## 6. Where it breaks

### 6.1. An effect without heterogeneity

The same data, but the effect is the same for everyone: $\tau(x)=1$. A good check should say that there is no heterogeneity, even though the forest will still give different objects different predictions.

### 6.2. Checking on the same data the model was trained on

All the checks above rely on the prediction $S$ coming from a different part of the sample. If BLP is computed on the same part $A$ where the forest was trained, the noise the forest has fitted enters both $S$ and the outcome.

**Task 5. Generate data with `simulate(4000, np.random.default_rng(1), heterogeneous=False)`. Compute the BLP $\beta_2$ with its standard error, the GATES $\gamma_1,\dots,\gamma_5$ and the AUTOC with its standard error on the main part $M$ (the forest and $B$ are fitted on $A$). Then compute the BLP $\beta_2$ on part $A$ itself: predictions `proxies(X0, T0, Y0, A, A)`. What do the checks show on $M$, and what on $A$?**

In [ ]:
# Your code here

**Your answer:**

## 7. The library call: `econml.validate.DRTester`

In EconML the same checks are collected in `DRTester`: `fit_nuisance` computes doubly robust scores on the main part with models fitted on the auxiliary part, `evaluate_blp` gives BLP, `evaluate_cal` a calibration by groups, `evaluate_uplift` a Qini or TOC with a bootstrap. They are built differently from the papers, and it shows in the numbers:

- `evaluate_blp` is an OLS regression of the doubly robust scores on the uncentered prediction, not the weighted outcome regression of Chernozhukov et al.; the slope estimates the same $\beta_2$, but with different auxiliary models;
- `evaluate_uplift` integrates the curve only over the prediction percentiles from 5 to 95 (the thresholds are percentiles of the prediction on the auxiliary part), not over the whole share from 0 to 1; with `metric='toc'` the integrand is $\mathrm{TOC}(u)$, and the library's AUTOC is smaller than ours, with `metric='qini'` it is $u\cdot\mathrm{TOC}(u)$, which is small at the ends, and the number almost matches our Qini;
- the standard errors of `evaluate_uplift` come from the influence function, while the bootstrap `n_bootstrap` is used only for a uniform confidence band of the curve.

**Task 6. Create `DRTester(model_regression=gb(), model_propensity=LogisticRegression(), cate=cf)`, call `fit_nuisance(X[M], T[M], Y[M], X[A], T[A], Y[A])` and print the slope of `evaluate_blp(X[M], X[A])`, the `cal_r_squared` of `evaluate_cal()` and the coefficients of `evaluate_uplift(metric='qini', n_bootstrap=200)` and `evaluate_uplift(metric='toc', n_bootstrap=200)` with their standard errors. Which library number matches ours to the third decimal?**

In [ ]:
# Your code here

**Your answer:**

## Implementations

- `econml.validate.DRTester` (EconML, MIT license) — BLP, calibration and Qini/TOC, as in section 7.
- The R package [grf](https://github.com/grf-labs/grf) (GPL-3.0 license): `test_calibration()` — a BLP-style test with coefficients on the mean and on the deviation of the forest's prediction, `rank_average_treatment_effect()` — RATE with AUTOC and Qini, the authors' implementation of Yadlowsky et al.
- The R package [GenericML](https://github.com/mwelz/GenericML), referenced by Chernozhukov et al., — BLP, GATES and CLAN with repeated splits. The paper splits the sample not once but many times (250 in its example) and reports the median of the estimates and p-values across splits: the result of one split depends on the random division. The notebook uses one split so that the numbers can be checked.

## What to take away

- **BLP answers whether there is heterogeneity and whether the prediction carries it,** with one coefficient $\beta_2$ and its interval.
- **GATES is calibration by quantiles with intervals,** CLAN describes the extreme groups by observed features.
- **RATE evaluates the prediction as a priority rule:** how much larger the effect is among those served first than on average.
- **Without heterogeneity the checks on the held-out part stay silent,** although the forest's predictions differ.
- **A check on the data the model was trained on lies:** fitted noise looks like strong heterogeneity.

Sources:
1. [Chernozhukov, Demirer, Duflo & Fernández-Val, 2017. Generic machine learning inference on heterogeneous treatment effects in randomized experiments](https://arxiv.org/abs/1712.04802)
2. [Yadlowsky, Fleming, Shah, Brunskill & Wager, 2021. Evaluating treatment prioritization rules via rank-weighted average treatment effects](https://arxiv.org/abs/2111.07966)
3. [EconML](https://github.com/py-why/EconML), [grf](https://github.com/grf-labs/grf)